# Semantic correspondence: Colab runner

This notebook only sets options and calls the repo code (`scripts/colab_setup.py`, `run.py`).
Code changes belong in the repo: edits made here are lost. Full guide: `RUN.md`.

**Before the first run:**
1. Runtime > Change runtime type > **GPU** (needed for `train`; `evaluate`/`submit` also work on CPU, slowly).
2. Accept the competition rules on Kaggle.
3. Add these to Colab Secrets (key icon) and allow notebook access: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `WANDB_API_KEY`, `HF_TOKEN`.

In [ ]:
# Run settings
BRANCH = "dinov3"        # branch to run; push your changes there first
MODE = "evaluate"        # "train" | "evaluate" | "submit"   (settings: BASE in src/config.py)
RUN = ""                 # evaluate/submit an existing run (uses its best.pt)
RESUME = ""              # train: continue this run id after a disconnect
EXTRA = ""               # other run.py flags, e.g. "--dry-run"

In [ ]:
# Check Hugging Face access to the gated DINOv3 weights before the setup downloads them.
# 403 / GatedRepoError: open https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m logged in with the
# account printed here and request access; wait until the page says access is granted.
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

HF_MODEL = "facebook/dinov3-vitb16-pretrain-lvd1689m"
hf_token = userdata.get("HF_TOKEN").strip()
me = HfApi(token=hf_token).whoami()
print("HF_TOKEN account:", me["name"], "| token role:", me.get("auth", {}).get("accessToken", {}).get("role"))
try:
    hf_hub_download(HF_MODEL, "config.json", token=hf_token)
    print("DINOv3 access OK")
except Exception as e:
    print(f"No access to {HF_MODEL}: {type(e).__name__}. Request access with the account above "
          "(fine-grained token: enable 'Read access to contents of all public gated repos you can access').")


In [ ]:
# Clone the repo the first time, then let scripts/colab_setup.py do the rest:
# pull, install environment.yml (pip part), secrets, Drive, data and weights.
import os

REPO_URL = "https://github.com/julpfi/sapienza_aml_semantic_correspondence.git"
REPO_DIR = "/content/sapienza_aml_semantic_correspondence"
if not os.path.isdir(f"{REPO_DIR}/.git"):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%run scripts/colab_setup.py --branch {BRANCH}

In [ ]:
# Optional quick edits of src/config.py in this session only (uncommitted, blocks a later git pull of
# config.py). Off for the DINOv3 run: it would change grad_clip and loss_temperature.
# !sed -i 's/"fusion": "concat",/"fusion": "adapter",/; s/"grad_clip": 10.0,/"grad_clip": 20.0,/; s/"loss_temperature": 0.04,/"loss_temperature": 0.02,/' s…
# !grep -nE '"fusion":|"grad_clip":|"loss_temperature"' src/config.py


In [ ]:
# Start the run. Runs are saved on your Drive: aml_semantic_correspondence/outputs/<run_id>
flags = f"--mode {MODE} {EXTRA}"
if RESUME:
    flags += f" --resume {RESUME}"
elif RUN:
    flags += f" --run {RUN}"
if MODE == "train":
    flags += " --require-gpu"
!python run.py {flags}

In [ ]:
# Optional: paired comparison of evaluated runs (seeds comma-separated, A then B), e.g. "a_s0,a_s1 b_s0,b_s1".
# Each run needs per_query.csv: MODE = "evaluate" with RUN = "<run_id>" writes it.
COMPARE = ""
if COMPARE:
    !python scripts/compare_runs.py {COMPARE}

In [ ]:
# Optional: upload a submission to Kaggle (set the run id printed above).
SUBMIT_RUN = "v1_confirmation"  # set to the run id you want to submit, or "" to skip
if SUBMIT_RUN:
    !python scripts/kaggle_io.py submit --file /content/drive/MyDrive/aml_semantic_correspondence/outputs/{SUBMIT_RUN}/submission.csv --message "{SUBMIT_RUN}"